**Quantum operators and Hermitian matrices.**
In quantum mechanics every *observable* (position, momentum, energy, spin, …)
is represented by a **Hermitian operator** $\hat{O}$.
A matrix $H$ is Hermitian when it equals its own conjugate transpose:

$$
H = H^\dagger \equiv (H^*)^T
$$

Hermitian matrices have two properties that make them physically meaningful:
their **eigenvalues are always real** (observable quantities must be real numbers),
and their **eigenvectors are mutually orthogonal** (distinct measurement outcomes
are distinguishable states).

In [ ]:
"""operators.ipynb"""

# Cell 01 - Create two random state vectors (not normalized)

import numpy as np
from IPython.display import Math, display

from qis101_utils import as_latex

np.random.seed(2016)
ndims = 5

psi = np.random.random(ndims) + np.random.random(ndims) * 1j
phi = np.random.random(ndims) + np.random.random(ndims) * 1j

display(as_latex(psi, prefix=r"\mathbf{\lvert\psi\rangle}=", column=True))
display(as_latex(phi, prefix=r"\mathbf{\lvert\phi\rangle}=", column=True))

**Constructing a Hermitian matrix.**
To build a random Hermitian matrix we fill the upper triangle with arbitrary
complex entries, place real values on the diagonal (a complex number $a + bi$
equals its own conjugate only when $b = 0$), then mirror each off-diagonal
entry to its transpose position with the imaginary part negated:

$$
H_{ij} = \overline{H_{ji}} \quad (i \neq j), \qquad H_{ii} \in \mathbb{R}
$$

In [ ]:
# Cell 02 - Create a random Hermitian operator matrix


def create_hermitian_matrix(ndims: int) -> np.ndarray:
    """Return a random ndims x ndims Hermitian matrix.

    Diagonal entries are real; off-diagonal entries satisfy H[i,j] = conj(H[j,i]).
    """
    a = np.zeros((ndims, ndims), dtype=complex)
    for i in range(ndims):
        for j in range(i, ndims):
            r1 = np.random.random()
            r2 = np.random.random()
            if i == j:
                a[i, j] = r1  # real diagonal
            else:
                a[i, j] = complex(r1, r2)
                a[j, i] = complex(r1, -r2)  # conjugate mirror
    return a


op = create_hermitian_matrix(ndims)

display(as_latex(op, prefix=r"\mathbf{\hat{O}}="))

**Eigenvalues and eigenvectors of a Hermitian operator.**
An *eigenket* $|v\rangle$ of operator $\hat{O}$ satisfies $\hat{O}|v\rangle = \lambda|v\rangle$,
where the scalar $\lambda$ is the corresponding eigenvalue.
For a Hermitian operator all eigenvalues are guaranteed to be real.

We use `numpy.linalg.eigh` rather than `eig` because `eigh` exploits
the Hermitian structure to guarantee real eigenvalues and orthonormal
eigenvectors in its return values, avoiding the small spurious imaginary
residuals that `eig` can produce from floating-point arithmetic.

The eigenvalue equation can be verified for any test vector $|\phi\rangle$
by sandwiching both sides with the bra $\langle\phi|$:

$$
\langle\phi|\hat{O}|v_i\rangle = \langle\phi|\lambda_i|v_i\rangle
$$

In [ ]:
# Cell 03 - Verify the eigenvalue equation for each eigenvector

# eigh is preferred over eig for Hermitian matrices:
# it returns strictly real eigenvalues and orthonormal eigenvectors.
eigen_vals, eigen_vecs = np.linalg.eigh(op)

display(as_latex(eigen_vals, prefix=r"\mathbf{\lambda}="))

# numpy returns eigenvectors as columns, so eigen_vecs[:, i] is the i-th eigenvector
for i in range(ndims):
    display(as_latex(eigen_vecs[:, i], prefix=rf"\mathbf{{v_{i}}}=", column=False))

# bra_phi is the conjugate of phi (no .T needed — phi is already 1-D)
bra_phi = phi.conj()

for i in range(ndims):
    lhs = np.dot(bra_phi, op @ eigen_vecs[:, i])
    rhs = np.dot(bra_phi, eigen_vals[i] * eigen_vecs[:, i])
    display(
        Math(
            rf"\langle\phi\lvert\hat{{O}}\lvert v_{i}\rangle="
            rf"\langle\phi\lvert\lambda_{i}\lvert v_{i}\rangle"
            rf"\;\rightarrow\;{np.isclose(lhs, rhs)}"
        )
    )

**Orthogonality of eigenvectors.**
Two vectors are orthogonal when their inner product is zero.
For a Hermitian operator with non-degenerate eigenvalues
(all $\lambda_i$ distinct), the eigenvectors are guaranteed to be
mutually orthogonal:

$$
\langle v_i | v_j \rangle = \overline{v_i} \cdot v_j = 0 \quad (i \neq j)
$$

This is what makes eigenvectors suitable as a basis — they span the space
without redundancy, and any state vector can be expressed as a unique
linear combination of them.

In [ ]:
# Cell 04 - Verify mutual orthogonality of all eigenvector pairs

for i in range(ndims):
    for j in range(i + 1, ndims):
        # inner = np.dot(eigen_vecs[:, i].conj(), eigen_vecs[:, j]).round(4).real + 0.0
        inner = np.dot(eigen_vecs[:, i].conj(), eigen_vecs[:, j]).round(4).real + 0.0
        display(Math(rf"\langle v_{i}\lvert v_{j}\rangle\;=\;{inner}"))

**Matrix representation in the eigenbasis.**
Any operator can be expressed as a matrix whose entries depend on the chosen basis.
In the operator's own eigenbasis the matrix representation is diagonal,
with the eigenvalues on the diagonal:

$$
O_{ij} = \langle v_i | \hat{O} | v_j \rangle = \lambda_j\,\delta_{ij}
$$

This is the quantum-mechanical analogue of diagonalizing a matrix —
the eigenbasis is the "natural" coordinate system in which the operator
acts simply by scaling each basis vector by its eigenvalue.

In [ ]:
# Cell 05 - Express the operator as a matrix in its own eigenbasis


def get_matrix_from_operator(op: np.ndarray, basis: list[np.ndarray]) -> np.ndarray:
    """Return the matrix representation of op in the given orthonormal basis.

    Each entry is the inner product  M[i,j] = <basis[i] | op | basis[j]>.

    Parameters
    ----------
    op    : ndarray, shape (n, n) — the operator matrix in the standard basis.
    basis : list of n column vectors, each of shape (n,).

    Returns
    -------
    ndarray, shape (n, n) — the operator matrix in the provided basis.
    """
    n = len(basis)
    m = np.zeros((n, n), dtype=complex)
    for i in range(n):
        for j in range(n):
            m[i, j] = np.dot(basis[i].conj(), op @ basis[j])
    return m


# Simple 2x2 Hermitian operator with known eigenvalues 6 and 2
op = np.array([[4, -2], [-2, 4]], dtype=complex)
eigen_vals, eigen_vecs = np.linalg.eigh(op)

# Build basis as a list of column eigenvectors (columns of eigen_vecs)
basis = [eigen_vecs[:, i] for i in range(op.shape[0])]

m = get_matrix_from_operator(op, basis)

display(as_latex(op, prefix=r"\mathbf{\hat{O}}="))
display(as_latex(eigen_vecs, prefix=r"\mathbf{\epsilon}="))
display(as_latex(np.round(m.real, 4), prefix=r"\mathbf{O}_{\text{eigenbasis}}="))
display(as_latex(eigen_vals, prefix=r"\mathbf{\lambda}="))

**The commutator and simultaneous observables.**
The commutator of two operators $\Omega_1$ and $\Omega_2$ is defined as:

$$
[\Omega_1,\,\Omega_2] = \Omega_1\Omega_2 - \Omega_2\Omega_1
$$

When $[\Omega_1,\Omega_2] = 0$ the operators **commute**: they share a common
eigenbasis, meaning both observables can be measured simultaneously with
arbitrary precision.
When $[\Omega_1,\Omega_2] \neq 0$ the operators are **incompatible** and
obey an uncertainty relation — the canonical example being position and momentum:
$[\hat{x},\hat{p}] = i\hbar$.

In [ ]:
# Cell 06 - Two random Hermitian operators generally do not commute

ndims = 3
omega_1 = create_hermitian_matrix(ndims)
omega_2 = create_hermitian_matrix(ndims)

commutator = omega_1 @ omega_2 - omega_2 @ omega_1

display(as_latex(omega_1, prefix=r"\mathbf{\Omega_1}="))
display(as_latex(omega_2, prefix=r"\mathbf{\Omega_2}="))
display(as_latex(commutator, prefix=r"\mathbf{[\Omega_1,\Omega_2]}="))
display(
    Math(
        rf"\mathbf{{\Omega_1}}\;\text{{and}}\;\mathbf{{\Omega_2}}\;"
        rf"\text{{commute}}\;\rightarrow\;{np.isclose(commutator, 0).all()}"
    )
)

**Diagonal matrices always commute.**
For diagonal matrices $D_1$ and $D_2$, multiplication is entry-wise along the diagonal:
$(D_1 D_2)_{ii} = (D_1)_{ii}(D_2)_{ii} = (D_2 D_1)_{ii}$, so $[D_1, D_2] = 0$
regardless of the diagonal entries.
Physically, operators that are simultaneously diagonal in the same basis
share eigenvectors and therefore represent compatible observables.

In [ ]:
# Cell 07 - All diagonal matrices commute with each other


def create_diagonal_matrix(ndims: int) -> np.ndarray:
    """Return a random ndims x ndims diagonal matrix with complex entries."""
    return np.diag(np.random.random(ndims) + np.random.random(ndims) * 1j)


omega_1 = create_diagonal_matrix(ndims)
omega_2 = create_diagonal_matrix(ndims)

commutator = omega_1 @ omega_2 - omega_2 @ omega_1

display(as_latex(omega_1, prefix=r"\mathbf{\Omega_1}="))
display(as_latex(omega_2, prefix=r"\mathbf{\Omega_2}="))
display(as_latex(commutator, prefix=r"\mathbf{[\Omega_1,\Omega_2]}="))
display(
    Math(
        rf"\mathbf{{\Omega_1}}\;\text{{and}}\;\mathbf{{\Omega_2}}\;"
        rf"\text{{commute}}\;\rightarrow\;{np.isclose(commutator, 0).all()}"
    )
)